In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = '../data'
IMG = '../app/images'
os.makedirs(IMG, exist_ok=True)

# Shared chart style for the website
C_DISM, C_ESC = '#8A9BA8', '#B5452B'   # dismissed / escalated
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.25,
                     'font.size': 11})

def save(name):
    plt.tight_layout()
    plt.savefig(f'{IMG}/{name}.png', bbox_inches='tight')
    plt.close()  # script mode: no windows, the chart is already saved to app/images
    print('Saved:', f'{IMG}/{name}.png')

# %% [markdown]
# ## Load data

# %%
tr_sig = pd.read_csv(f'{DATA}/train_signals.csv', parse_dates=['signal_sanasi'])
te_sig = pd.read_csv(f'{DATA}/test_signals.csv', parse_dates=['signal_sanasi'])
tr_tx = pd.read_parquet(f'{DATA}/train_transactions.parquet')
tr_tx['tranzaksiya_vaqti'] = pd.to_datetime(tr_tx['tranzaksiya_vaqti'])

# Attach target and signal date to each transaction
tx = tr_tx.merge(tr_sig, on='signal_id', how='left')
tx['days_before'] = (tx['signal_sanasi'] + pd.Timedelta(days=1) - tx['tranzaksiya_vaqti']).dt.total_seconds() / 86400

print('Train alerts:', len(tr_sig), '| test alerts:', len(te_sig))
print('Train transactions:', len(tr_tx))
print('Median transactions per alert:', tr_tx.groupby('signal_id').size().median())
print(tx.head())

# %% [markdown]
# ## 1. Target distribution

# %%
counts = tr_sig['eskalatsiya'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Dismissed (0)', 'Escalated (1)'], counts.values, color=[C_DISM, C_ESC])
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f'{v:,}\n({v / counts.sum():.1%})', ha='center', va='bottom')
ax.set_title('Target distribution (train)'); ax.set_ylabel('Alerts')
save('01_target_distribution')
print(f"Escalation rate: {tr_sig['eskalatsiya'].mean():.2%}")

# %% [markdown]
# ✍️ **Insight:** _Are the classes balanced? What is the escalation rate?_

# %% [markdown]
# ## 2. Alerts over time

# %%
fig, ax1 = plt.subplots(figsize=(10, 4))
w_tr = tr_sig.set_index('signal_sanasi').resample('W')['eskalatsiya'].agg(['size', 'mean'])
w_te = te_sig.set_index('signal_sanasi').resample('W').size()
ax1.plot(w_tr.index, w_tr['size'], label='Train: alerts per week', color='#2F5D7C')
ax1.plot(w_te.index, w_te.values, label='Test: alerts per week', color='#9C7A3C', linestyle='--')
ax1.set_ylabel('Alerts per week')
ax2 = ax1.twinx()
ax2.plot(w_tr.index, w_tr['mean'], color=C_ESC, alpha=0.7, label='Escalation rate (train)')
ax2.set_ylabel('Escalation rate'); ax2.grid(False)
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=9)
ax1.set_title('Weekly alerts: train vs test')
save('02_signals_over_time')
print('Train:', tr_sig.signal_sanasi.min().date(), '—', tr_sig.signal_sanasi.max().date())
print('Test: ', te_sig.signal_sanasi.min().date(), '—', te_sig.signal_sanasi.max().date())

# %% [markdown]
# ✍️ **Insight:** _Do train and test periods overlap? Does the escalation rate change over time?_

# %% [markdown]
# ## 3. Transaction activity over time

# %%
m = tx.groupby(tx['tranzaksiya_vaqti'].dt.to_period('M')).size()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(m.index.to_timestamp(), m.values, color='#2F5D7C', marker='o', markersize=3)
ax.set_title('Transactions per month (train)'); ax.set_ylabel('Transactions')
save('03_tx_over_time')

# %% [markdown]
# ✍️ **Insight:** _Growth, seasonality, gaps?_

# %% [markdown]
# ## 4. Direction and transaction type

# %%
ct = pd.crosstab(tx['tranzaksiya_turi'], tx['kirim_chiqim'])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ct.plot(kind='bar', ax=axes[0], color=['#2F5D7C', '#9C7A3C'], rot=0)
axes[0].set_title('Transactions: type × direction'); axes[0].set_xlabel('')

share = (tx.groupby(['eskalatsiya', 'tranzaksiya_turi']).size()
           .groupby(level=0).transform(lambda s: s / s.sum()).unstack(0))
share.plot(kind='bar', ax=axes[1], color=[C_DISM, C_ESC], rot=0)
axes[1].set_title('Transaction type mix by alert outcome'); axes[1].set_xlabel('')
axes[1].legend(['Dismissed', 'Escalated'])
save('04_direction_type')
print(share.round(3))

# %% [markdown]
# ✍️ **Insight:** _Which types are more common for escalated alerts (e.g. naqd, xalqaro)?_

# %% [markdown]
# ## 5. Transaction size (`miqdor_indeksi`)

# %%
smp = tx.sample(min(len(tx), 500_000), random_state=42)
lo, hi = smp['miqdor_indeksi'].quantile([0.005, 0.995])
bins = np.linspace(lo, hi, 80)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for t, g in smp.groupby('tranzaksiya_turi'):
    axes[0].hist(g['miqdor_indeksi'], bins=bins, histtype='step', density=True, label=t, linewidth=1.5)
axes[0].set_title('By transaction type'); axes[0].legend(fontsize=9)
for y, col, lab in [(0, C_DISM, 'Dismissed'), (1, C_ESC, 'Escalated')]:
    axes[1].hist(smp.loc[smp.eskalatsiya == y, 'miqdor_indeksi'], bins=bins, histtype='step',
                 density=True, color=col, label=lab, linewidth=1.8)
axes[1].set_title('By alert outcome'); axes[1].legend()
for a in axes: a.set_xlabel('miqdor_indeksi')
save('05_amount_distribution')
print(tx.groupby('eskalatsiya')['miqdor_indeksi'].describe().round(3))

# %% [markdown]
# ✍️ **Insight:** _Do escalated alerts have different transaction sizes? Any outliers?_

# %% [markdown]
# ## 6. Activity before the alert
# Average number of transactions per alert for each day before the alert date. The most important chart for modelling.

# %%
win = tx[(tx['days_before'] >= 0) & (tx['days_before'] < 90)].copy()
win['day'] = win['days_before'].astype(int)
n_sig = tr_sig['eskalatsiya'].value_counts()
act = win.groupby(['eskalatsiya', 'day']).size().unstack(0).fillna(0) / n_sig
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(act.index, act[0], color=C_DISM, label='Dismissed', linewidth=2)
ax.plot(act.index, act[1], color=C_ESC, label='Escalated', linewidth=2)
ax.invert_xaxis()
ax.set_xlabel('Days before alert'); ax.set_ylabel('Transactions per alert per day')
ax.set_title('Activity during the 90 days before the alert'); ax.legend()
save('06_activity_before_signal')
print('Share of last-7-day transactions within the 90-day window:')
print((win[win.day < 7].groupby('eskalatsiya').size() / win.groupby('eskalatsiya').size()).round(3))

# %% [markdown]
# ✍️ **Insight:** _Does activity rise before the alert? Is the rise stronger for escalated alerts?_

# %% [markdown]
# ## 7. Escalated vs dismissed alerts

# %%
g = tx.groupby('signal_id')
feat = pd.DataFrame({
    'Transactions': g.size(),
    'Mean miqdor_indeksi': g['miqdor_indeksi'].mean(),
    'Max miqdor_indeksi': g['miqdor_indeksi'].max(),
    'Cash (naqd) share': g['tranzaksiya_turi'].apply(lambda s: (s == 'naqd').mean()),
    'International share': g['tranzaksiya_turi'].apply(lambda s: (s == 'xalqaro').mean()),
    'Outgoing share': g['kirim_chiqim'].apply(lambda s: (s == 'chiqim').mean()),
    'Transactions, last 7 days': tx[tx.days_before.between(0, 7)].groupby('signal_id').size(),
    'Days since last transaction': g['days_before'].min(),
}).fillna({'Transactions, last 7 days': 0})
feat = feat.join(tr_sig.set_index('signal_id')['eskalatsiya'])

cols = [c for c in feat.columns if c != 'eskalatsiya']
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, c in zip(axes.ravel(), cols):
    data = [feat.loc[feat.eskalatsiya == k, c].dropna() for k in (0, 1)]
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, medianprops={'color': '#1C2630'})
    ax.set_xticks([1, 2]); ax.set_xticklabels(['Dismissed', 'Escalated'])
    for patch, col in zip(bp['boxes'], [C_DISM, C_ESC]): patch.set_facecolor(col); patch.set_alpha(0.6)
    ax.set_title(c, fontsize=10)
fig.suptitle('Escalated vs dismissed alerts', fontsize=13)
save('07_class_comparison')
print(feat.groupby('eskalatsiya').median().T.round(3))

# %% [markdown]
# ✍️ **Insight:** _Which features differ most between the classes? These are modelling ideas._

# %% [markdown]
# ## 8. Hour of day and day of week

# %%
tx['hour'] = tx['tranzaksiya_vaqti'].dt.hour
tx['dow'] = tx['tranzaksiya_vaqti'].dt.dayofweek
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for y, col, lab in [(0, C_DISM, 'Dismissed'), (1, C_ESC, 'Escalated')]:
    s = tx[tx.eskalatsiya == y]
    axes[0].plot(s['hour'].value_counts(normalize=True).sort_index(), color=col, label=lab, linewidth=2)
    axes[1].plot(s['dow'].value_counts(normalize=True).sort_index(), color=col, label=lab, linewidth=2, marker='o')
axes[0].set_title('Share of transactions by hour'); axes[0].set_xlabel('Hour')
axes[1].set_title('Share of transactions by weekday'); axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
for a in axes: a.legend()
save('08_hour_weekday')

# %% [markdown]
# ✍️ **Insight:** _Any night-time activity? Do the classes differ?_ If every timestamp is 00:00, the data has no time of day and this chart can be removed.

# %% [markdown]
# ## Key findings for the website
# Once everything is filled in, list the 5–7 most important findings here and send them to Member 3:
# 
# 1. ...
# 2. ...
# 3. ...